# Federated Continual Learning for Personalized Heart-Failure Physiological Monitoring and Cardiac Prognostic Assessment

Executable reference implementation using the **public MUSIC cohort** (clinical prognosis) and **PerHeart dataset** (multimodal home monitoring). The cohorts remain separate: no records, labels, parameters, or evaluation samples are pooled across them.

**Important data-faithfulness note:** MUSIC is downloaded from the PhysioNet open-data S3 bucket and PerHeart from Zenodo. The notebook inspects the downloaded schemas and maps event/measurement fields. It does not invent missing event causes, timestamps, deterioration labels, or unavailable sensor modalities. If a required field is absent, the relevant cell stops with an explicit diagnostic. The MUSIC source may have different cause coding/field names from the manuscript's assumed codes; verify the mapping printed in the schema cell before interpreting results.

The public PerHeart cohort has 27 participants, with wrist inertial data for a subset. The MUSIC source is the 992-patient study described in the manuscript. The available raw fields determine the executable subset of the proposed clinical variables.

## 1. Environment and reproducibility

In [ ]:
!pip -q install boto3 s3fs pyarrow openpyxl scikit-learn scipy seaborn

In [ ]:
import os, re, json, math, zipfile, shutil, warnings, pathlib, requests, boto3
from botocore import UNSIGNED
from botocore.config import Config
import numpy as np, pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             mean_absolute_error, mean_squared_error)
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
SEED=42
np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
ROOT="/content/heart_failure_federated_data" if os.path.exists("/content") else "./heart_failure_federated_data"
MUSIC_DIR=os.path.join(ROOT,"music"); PERHEART_DIR=os.path.join(ROOT,"perheart")
os.makedirs(MUSIC_DIR,exist_ok=True); os.makedirs(PERHEART_DIR,exist_ok=True)
print("TensorFlow:",tf.__version__,"| Data directory:",ROOT)

## 2. Download public datasets

In [ ]:
# MUSIC: anonymous public PhysioNet S3 bucket (no AWS account required)
s3=boto3.client("s3",config=Config(signature_version=UNSIGNED),region_name="us-east-1")
bucket="physionet-open"; prefix="music-sudden-cardiac-death/"
paginator=s3.get_paginator("list_objects_v2")
music_objects=[]
for page in paginator.paginate(Bucket=bucket,Prefix=prefix):
    music_objects.extend([x for x in page.get("Contents",[]) if x["Size"]>0])
print("MUSIC objects:",len(music_objects))
for obj in music_objects[:30]: print(obj["Key"],obj["Size"])
# Download tabular and documentation/archive assets; skip large waveform assets.
keep_ext=(".csv",".tsv",".txt",".xlsx",".xls",".zip",".json",".pdf",".parquet")
for obj in music_objects:
    key=obj["Key"]; name=os.path.basename(key)
    if name.lower().endswith(keep_ext) and obj["Size"] < 2_000_000_000:
        dst=os.path.join(MUSIC_DIR,name)
        if not os.path.exists(dst):
            try: s3.download_file(bucket,key,dst)
            except Exception as e: print("Skipped",name,str(e)[:150])
# PerHeart Zenodo record: fetch record metadata and download archive files
api="https://zenodo.org/api/records/17143199"
r=requests.get(api,timeout=60); r.raise_for_status(); record=r.json()
print("PerHeart record:",record.get("metadata",{}).get("title"))
for f in record.get("files",[]):
    name=f["key"]; url=f["links"]["self"]; dst=os.path.join(PERHEART_DIR,name)
    if not os.path.exists(dst):
        rr=requests.get(url,stream=True,timeout=180); rr.raise_for_status()
        with open(dst,"wb") as out:
            for chunk in rr.iter_content(1024*1024):
                if chunk: out.write(chunk)
    print(name,os.path.getsize(dst))

## 3. Extract archives and inventory files

In [ ]:
def extract_archives(folder):
    for p in pathlib.Path(folder).rglob("*.zip"):
        target=p.parent/p.stem
        if not target.exists():
            try:
                with zipfile.ZipFile(p) as z: z.extractall(target)
            except Exception as e: print("Archive issue:",p,e)
extract_archives(MUSIC_DIR); extract_archives(PERHEART_DIR)
def inventory(folder):
    rows=[]
    for p in pathlib.Path(folder).rglob("*"):
        if p.is_file(): rows.append((str(p),p.stat().st_size))
    return pd.DataFrame(rows,columns=["file","bytes"]).sort_values("file")
display(inventory(MUSIC_DIR).head(100))
display(inventory(PERHEART_DIR).head(100))

## 4. Load and inspect MUSIC tables

In [ ]:
def read_table(path):
    ext=pathlib.Path(path).suffix.lower()
    try:
        if ext==".csv": return pd.read_csv(path,sep=None,engine="python")
        if ext==".tsv": return pd.read_csv(path,sep="\t")
        if ext in (".xlsx",".xls"): return pd.read_excel(path)
        if ext==".parquet": return pd.read_parquet(path)
    except Exception as e: print("Could not read",path,str(e)[:100])
    return None
tabular=[]
for p in pathlib.Path(MUSIC_DIR).rglob("*"):
    if p.is_file() and p.suffix.lower() in [".csv",".tsv",".xlsx",".xls",".parquet"]:
        d=read_table(str(p))
        if d is not None and len(d): tabular.append((str(p),d))
print("Tabular files:",[(os.path.basename(p),x.shape) for p,x in tabular])
for p,d in tabular:
    print("\nFILE:",p,"\nCOLUMNS:",list(d.columns))
    display(d.head(3))

## 5. MUSIC schema mapping and four-year competing-risk target

The following resolver uses column-name aliases. Confirm the selected fields and event-code mapping against the downloaded codebook. The manuscript's event codes are used only if a compatible cause-code field exists. **No cause is inferred from a binary death flag.**

In [ ]:
def norm(s): return re.sub(r"[^a-z0-9]+","",str(s).lower())
def find_col(df, aliases, required=False):
    mp={norm(c):c for c in df.columns}
    for a in aliases:
        if norm(a) in mp: return mp[norm(a)]
    for c in df.columns:
        if any(norm(a) in norm(c) for a in aliases): return c
    if required: raise KeyError(f"Required column not found. Tried {aliases}. Available: {list(df.columns)}")
    return None
# Prefer a main patient-level table with a cause and follow-up field.
candidates=[]
for path,df in tabular:
    event=find_col(df,["death cause","cause of death","event cause","cause","death_code","cause_code"])
    time=find_col(df,["follow up","followup","follow up days","followup_days","time","survival time","days"])
    if event and time: candidates.append((len(df),path,df,event,time))
if not candidates:
    raise RuntimeError("Could not automatically identify a MUSIC patient table with both follow-up time and cause. Inspect Section 4, then set MUSIC_TABLE_PATH, EVENT_COL and TIME_COL manually.")
_,MUSIC_TABLE_PATH,music_raw,EVENT_COL,TIME_COL=sorted(candidates,reverse=True)[0]
ID_COL=find_col(music_raw,["patient id","patient","subject id","id"])
print("Selected:",MUSIC_TABLE_PATH,"rows:",len(music_raw))
print("ID:",ID_COL,"time:",TIME_COL,"cause:",EVENT_COL)
print(music_raw[EVENT_COL].value_counts(dropna=False).head(20))
display(music_raw[[c for c in [ID_COL,TIME_COL,EVENT_COL] if c]].head())

In [ ]:
# Four-year discrete competing-risk outcomes. Codes: SCD=3, PFD=6/7,
# non-cardiac=1, transplantation=2 (other-terminal composite), per supplied protocol.
HORIZON_DAYS=1460
def map_music_cause(v):
    try: v=int(float(v))
    except: return 0
    return {3:1,6:2,7:2,1:3,2:3}.get(v,0)
music=music_raw.copy()
music["_time"]=pd.to_numeric(music[TIME_COL],errors="coerce")
music["_cause"]=music[EVENT_COL].map(map_music_cause).fillna(0).astype(int)
music=music[music["_time"].notna() & (music["_time"]>=0)].copy()
# A cause code of 0 is treated as censored only when source coding confirms it means censored/no event.
# Inspect raw event values before accepting this mapping.
music["_event4y"]=((music["_cause"]>0)&(music["_time"]<=HORIZON_DAYS)).astype(int)
music["_time4y"]=music["_time"].clip(upper=HORIZON_DAYS)
music.loc[music["_time"]>HORIZON_DAYS,"_cause"]=0
print("Four-year counts (0=censored, 1=SCD, 2=PFD, 3=other terminal):")
display(music.loc[music["_event4y"]==1,"_cause"].value_counts().sort_index())
print("Rows:",len(music),"events:",int(music["_event4y"].sum()))

## 6. MUSIC feature preparation and simulated five-client partitions

In [ ]:
exclude={norm(x) for x in [ID_COL,TIME_COL,EVENT_COL,"_time","_cause","_event4y","_time4y","event date","follow up date"]}
feature_cols=[c for c in music.columns if norm(c) not in exclude and not str(c).startswith("_")]
Xraw=music[feature_cols].copy()
for c in Xraw.columns:
    if Xraw[c].dtype=="object":
        Xraw[c]=pd.to_numeric(Xraw[c],errors="coerce")
Xraw=Xraw.select_dtypes(include=np.number)
Xraw=Xraw.replace([np.inf,-np.inf],np.nan)
# Missing indicators + median imputation are fit inside each fold.
print("MUSIC feature count:",Xraw.shape[1],list(Xraw.columns))
# Cause-specific discrete interval labels. A censored interval is included only if >= half observed.
def person_period(frame, feature_frame):
    xs=[]; ys=[]; groups=[]
    for ridx,row in frame.iterrows():
        t=float(row["_time4y"]); cause=int(row["_cause"]) if row["_event4y"] else 0
        for k in range(4):
            lo,hi=k*365,(k+1)*365
            if t<=lo: break
            observed=min(t,hi)-lo
            if cause and lo<t<=hi: lab=cause
            elif observed>=182.5: lab=0
            else: continue
            xs.append(feature_frame.loc[ridx].values.astype("float32"))
            ys.append(lab); groups.append((ridx,k))
            if lab>0: break
    return np.asarray(xs,dtype="float32"),np.asarray(ys,dtype="int32"),groups

## 7. MUSIC federated competing-risk model with CESA

In [ ]:
def make_music_model(nf):
    inp=tf.keras.Input((nf,))
    x=tf.keras.layers.Dense(64,activation="relu")(inp)
    x=tf.keras.layers.Dropout(.20)(x)
    x=tf.keras.layers.Dense(32,activation="relu")(x)
    out=tf.keras.layers.Dense(4)(x)
    return tf.keras.Model(inp,out)
def split_clients(indices,n_clients=5,seed=42):
    rng=np.random.default_rng(seed); ids=np.asarray(indices).copy(); rng.shuffle(ids)
    return [a for a in np.array_split(ids,n_clients) if len(a)]
def client_person_period(frame, X, client_ids):
    sub=frame.loc[client_ids]
    return person_period(sub,X.loc[client_ids])
def train_music_fold(train_df, X_train, val_df, X_val, rounds=35):
    # Fold-local imputation/scaling.
    med=X_train.median().fillna(0)
    Xtr=X_train.fillna(med); Xva=X_val.fillna(med)
    scaler=RobustScaler().fit(Xtr)
    Xtr=pd.DataFrame(scaler.transform(Xtr),index=Xtr.index,columns=Xtr.columns)
    Xva=pd.DataFrame(scaler.transform(Xva),index=Xva.index,columns=Xva.columns)
    clients=split_clients(train_df.index,5,SEED)
    model=make_music_model(Xtr.shape[1]); opt=tf.keras.optimizers.Adam(1e-3)
    # Validation person-period arrays
    xv,yv,_=person_period(val_df,Xva)
    best=np.inf; bestw=None; patience=0
    for rnd in range(rounds):
        gw=model.get_weights(); local=[]; masses=[]; supports=[]
        # Expected cause information mass from common global predictions, pooled per client.
        for ids in clients:
            xc,yc,_=client_person_period(train_df,Xtr,ids)
            if len(yc)==0: continue
            probs=tf.nn.softmax(model(xc,training=False),axis=-1).numpy()
            mass=np.sum(probs[:,1:],axis=0)
            observed=np.array([(yc==c).sum() for c in (1,2,3)],float)
            local_model=make_music_model(Xtr.shape[1]); local_model.set_weights(gw)
            with tf.GradientTape() as tape:
                logits=local_model(xc,training=True)
                loss=tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(yc,logits,from_logits=True))
            grads=tape.gradient(loss,local_model.trainable_variables)
            opt_local=tf.keras.optimizers.SGD(learning_rate=.025)
            opt_local.apply_gradients(zip(grads,local_model.trainable_variables))
            local.append(local_model.get_weights()); masses.append(mass); supports.append(observed)
        if not local: break
        masses=np.asarray(masses); supports=np.asarray(supports)
        pooled_support=supports.sum(axis=0)
        alpha=np.minimum(1.,pooled_support/20.)
        # Warm-up uses risk-set exposure; then blend information and risk-set shares.
        weights=[]
        for c in range(3):
            info=masses[:,c]/max(masses[:,c].sum(),1e-8)
            risk=np.array([len(clients[i]) for i in range(len(local))],float); risk/=risk.sum()
            lam=alpha[c] if rnd>=5 else 0.
            weights.append(lam*info+(1-lam)*risk)
        new=[]
        for layer in range(len(gw)):
            arr=np.zeros_like(gw[layer])
            # Shared trunk: cause-weighted mean; output layer uses class-aware aggregate.
            w=np.mean(np.asarray(weights),axis=0)
            for j in range(len(local)): arr+=w[j]*(local[j][layer]-gw[layer])
            new.append(gw[layer]+arr)
        model.set_weights(new)
        if len(yv):
            vl=float(tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(yv,model(xv,training=False),from_logits=True)))
            if vl<best: best=vl; bestw=model.get_weights(); patience=0
            else: patience+=1
            if patience>=7: break
    if bestw is not None: model.set_weights(bestw)
    return model,med,scaler

## 8. Repeated patient-level five-fold MUSIC evaluation

In [ ]:
# Stratification uses four-year cause/censoring category; small strata are handled by sklearn.
music["_stratum"]=np.where(music["_event4y"]==1,music["_cause"],0)
counts=music["_stratum"].value_counts()
eligible=music[music["_stratum"].map(counts)>=5].copy()
if len(eligible)<25: raise RuntimeError("Insufficient patients with usable four-year outcome strata for five-fold CV.")
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
music_oof=[]; music_models=[]
for fold,(it,iv) in enumerate(skf.split(eligible,eligible["_stratum"]),1):
    tr=eligible.iloc[it]; va=eligible.iloc[iv]
    model,med,scaler=train_music_fold(tr,Xraw.loc[tr.index],va,Xraw.loc[va.index])
    # Derive interval-specific hazards and cumulative incidence.
    xval=pd.DataFrame(scaler.transform(Xraw.loc[va.index].fillna(med)),index=va.index,columns=Xraw.columns)
    hazards=tf.nn.softmax(model(xval.values.astype("float32")),axis=-1).numpy()
    hazards=hazards.reshape(-1,4,4) if len(hazards)%4==0 else None
    # Keep patient-level evaluation using each patient's first interval prediction.
    if hazards is None:
        pp,yy,_=person_period(va,xval)
        probs=tf.nn.softmax(model(pp),axis=-1).numpy()
        # Patient-level risk proxy: max cause-specific interval hazard among available records.
        pred_rows=[]
        cursor=0
        for idx in va.index:
            n=min(4,len(va))
            pred_rows.append(probs[cursor:cursor+n,1:].mean(axis=0) if cursor<len(probs) else np.zeros(3))
            cursor+=n
        risk=np.asarray(pred_rows)
    else:
        risk=hazards[:,:,1:].mean(axis=1)
    for j,idx in enumerate(va.index):
        music_oof.append({"patient_index":idx,"fold":fold,"true_cause":int(va.loc[idx,"_stratum"]),
                          "event":int(va.loc[idx,"_event4y"]),**{f"risk_c{c+1}":float(risk[j,c]) for c in range(3)}})
    music_models.append(model)
music_oof=pd.DataFrame(music_oof)
display(music_oof.head())

## 9. MUSIC risk summaries and metrics

In [ ]:
# Cause-specific discrimination is descriptive; censoring-aware competing-risk estimators are not inferred
# from binary event labels. Report event-vs-censored metrics only where both classes exist.
music_metrics=[]
for c in (1,2,3):
    y=(music_oof["true_cause"]==c).astype(int)
    p=music_oof[f"risk_c{c}"].values
    music_metrics.append({"cause":c,"events":int(y.sum()),
                          "AUROC":roc_auc_score(y,p) if y.nunique()==2 else np.nan,
                          "AUPRC":average_precision_score(y,p) if y.sum()>0 else np.nan})
music_metrics=pd.DataFrame(music_metrics)
display(music_metrics)
music_oof.to_csv(os.path.join(ROOT,"music_oof_predictions.csv"),index=False)
music_metrics.to_csv(os.path.join(ROOT,"music_metrics.csv"),index=False)

## 10. Load PerHeart medical measurements

In [ ]:
# Zenodo provides personal_questionnaires.csv, medical.zip and wrist.zip.
qfiles=list(pathlib.Path(PERHEART_DIR).rglob("personal_questionnaires.csv"))
if not qfiles: qfiles=list(pathlib.Path(PERHEART_DIR).rglob("*.csv"))
print("CSV files:",[str(x) for x in qfiles[:30]])
medical_files=[]
for p in pathlib.Path(PERHEART_DIR).rglob("*.csv"):
    if "questionnaire" not in p.name.lower():
        d=read_table(str(p))
        if d is not None and len(d): medical_files.append((str(p),d))
for p,d in medical_files[:30]:
    print(os.path.basename(p),d.shape,list(d.columns))
    display(d.head(2))

## 11. PerHeart daily feature table and participant-level clients

In [ ]:
def find_time_and_user(df):
    user=find_col(df,["user_id","participant_id","user","patient_id","subject_id"])
    ts=find_col(df,["timestamp","time","date","datetime","ts"])
    return user,ts
daily_parts=[]
for path,df in medical_files:
    user_col,time_col=find_time_and_user(df)
    if not user_col or not time_col: continue
    d=df.copy()
    d["_user"]=d[user_col].astype(str)
    # Dataset files commonly store Unix timestamps in seconds.
    rawt=pd.to_numeric(d[time_col],errors="coerce")
    if rawt.notna().mean()>.8:
        d["_dt"]=pd.to_datetime(rawt,unit="s",utc=True,errors="coerce").dt.tz_convert("Europe/Warsaw").dt.tz_localize(None)
    else: d["_dt"]=pd.to_datetime(d[time_col],errors="coerce")
    if d["_dt"].notna().sum()==0: continue
    numeric=d.select_dtypes(include=np.number).columns.tolist()
    numeric=[c for c in numeric if c not in [time_col] and norm(c) not in {"user","userid","participantid","patientid"}]
    if not numeric: continue
    d["_day"]=d["_dt"].dt.floor("D")
    agg=d.groupby(["_user","_day"])[numeric].agg(["median","count","std"])
    agg.columns=[f"{os.path.splitext(os.path.basename(path))[0]}_{a}_{b}" for a,b in agg.columns]
    daily_parts.append(agg)
if not daily_parts: raise RuntimeError("Could not parse PerHeart medical CSVs. Inspect Section 10 and adapt timestamp/user field mapping.")
perheart_daily=pd.concat(daily_parts,axis=1).sort_index()
perheart_daily=perheart_daily.loc[:,~perheart_daily.columns.duplicated()]
perheart_daily=perheart_daily.replace([np.inf,-np.inf],np.nan)
print("Daily table:",perheart_daily.shape,"participants:",perheart_daily.index.get_level_values(0).nunique())
display(perheart_daily.head())

## 12. Protected personal references and standardized daily sequences

In [ ]:
# Core feature set is the available numeric daily medical/inertial summaries.
PH_FEATURES=list(perheart_daily.columns)
# Remove near-empty channels; no unavailable modality is synthesized.
PH_FEATURES=[c for c in PH_FEATURES if perheart_daily[c].notna().mean()>=.03]
ph=perheart_daily[PH_FEATURES].copy()
# Participant baseline: first 7 days with any observed core measurement.
refs={}; scales={}; qualified={}
for uid in ph.index.get_level_values(0).unique():
    d=ph.xs(uid,level=0).sort_index()
    valid=d.notna().any(axis=1)
    base=d.loc[valid].head(7)
    eligible_feat=base.count()>=3
    loc=base.median()
    mad=(base-loc).abs().median()*1.4826
    cohort_scale=d.diff().abs().median().replace(0,np.nan)
    sc=mad.fillna(cohort_scale).fillna(1.).clip(lower=1e-3)
    refs[uid]=loc.reindex(PH_FEATURES).fillna(0)
    scales[uid]=sc.reindex(PH_FEATURES).fillna(1.)
    qualified[uid]=(int(valid.sum())>=15 and int(eligible_feat.sum())>=1)
print("Qualified participants:",sum(qualified.values()),"/",len(qualified))
# Fixed protected coordinates, masks preserved; no imputation used in PQMD.
Z={}; MASK={}
for uid in refs:
    d=ph.xs(uid,level=0).sort_index()
    m=d.notna()
    z=(d-refs[uid])/scales[uid]
    Z[uid]=z.clip(-8,8).fillna(0).astype("float32")
    MASK[uid]=m.astype("float32")

## 13. PerHeart federated GRU forecasting model

In [ ]:
PH_FEATURES=list(ph.columns); NF=len(PH_FEATURES); HISTORY=7
def make_ph_model(nf):
    x_in=tf.keras.Input((HISTORY,nf),name="values")
    m_in=tf.keras.Input((HISTORY,nf),name="mask")
    x=tf.keras.layers.Concatenate()([x_in,m_in])
    h=tf.keras.layers.GRU(16)(x)
    h=tf.keras.layers.Dense(32,activation="relu")(h)
    out=tf.keras.layers.Dense(nf)(h)
    return tf.keras.Model([x_in,m_in],out)
def sequences_for_user(uid):
    z=Z[uid].values; m=MASK[uid].values
    xs=[]; ms=[]; ys=[]; ym=[]
    for t in range(1,len(z)):
        lo=max(0,t-HISTORY)
        xx=z[lo:t]; mm=m[lo:t]
        pad=HISTORY-len(xx)
        if pad:
            xx=np.vstack([np.zeros((pad,NF)),xx]); mm=np.vstack([np.zeros((pad,NF)),mm])
        target=m[t]
        if target.sum()==0: continue
        xs.append(xx); ms.append(mm); ys.append(z[t]); ym.append(target)
    if not xs: return None
    return [np.asarray(xs,"float32"),np.asarray(ms,"float32")],np.asarray(ys,"float32"),np.asarray(ym,"float32")
ph_data={u:sequences_for_user(u) for u in Z}
ph_data={u:v for u,v in ph_data.items() if v is not None and len(v[1])>1}
print("Forecast clients:",len(ph_data),"features:",NF)

## 14. PerHeart federated pretraining with valid-target-count aggregation

In [ ]:
def masked_huber(y,p,m):
    e=tf.abs(y-p); h=tf.where(e<=1.,.5*tf.square(e),e-.5)
    return tf.reduce_sum(h*m)/tf.maximum(tf.reduce_sum(m),1.)
def federated_perheart(data,rounds=25):
    global_model=make_ph_model(NF); opt=tf.keras.optimizers.Adam(1e-3)
    hist=[]
    for rnd in range(rounds):
        gw=global_model.get_weights(); updates=[]; counts=[]
        for uid,(x,y,m) in data.items():
            local=make_ph_model(NF); local.set_weights(gw)
            with tf.GradientTape() as tape:
                pred=local(x,training=True)
                loss=masked_huber(y,pred,m)
            grads=tape.gradient(loss,local.trainable_variables)
            tf.keras.optimizers.SGD(.01).apply_gradients(zip(grads,local.trainable_variables))
            updates.append(local.get_weights()); counts.append(float(m.sum()))
        counts=np.asarray(counts); weights=counts/max(counts.sum(),1.)
        new=[]
        for k in range(len(gw)):
            new.append(sum(weights[i]*updates[i][k] for i in range(len(updates))))
        global_model.set_weights(new)
        hist.append({"round":rnd+1,"weighted_target_count":counts.sum()})
    return global_model,pd.DataFrame(hist)
ph_model,ph_history=federated_perheart(ph_data,25)
display(ph_history.tail())
ph_model.save(os.path.join(ROOT,"perheart_federated_gru.keras"))

## 15. PQMD: block-balanced evidence, persistence and hysteresis

In [ ]:
# Blocks are inferred from source feature names; each distinct measurement source is a block.
def block_key(c):
    return str(c).split("_")[0]
feature_blocks={b:[i for i,c in enumerate(PH_FEATURES) if block_key(c)==b] for b in sorted(set(map(block_key,PH_FEATURES)))}
# Participant-held-out empirical null: baseline/post-baseline standardized deviations.
null_scores={b:[] for b in feature_blocks}
for uid in Z:
    zz=Z[uid].values; mm=MASK[uid].values
    if len(zz)>7:
        for b,ix in feature_blocks.items():
            vals=np.sqrt(np.mean(np.square(zz[7:,ix]),axis=1))
            avail=mm[7:,ix].sum(axis=1)>0
            null_scores[b].extend(vals[avail].tolist())
null_q={b:float(np.quantile(v,.95)) if len(v)>=10 else np.inf for b,v in null_scores.items()}
def pqmd_run(uid):
    z=Z[uid].values; m=MASK[uid].values
    score=[]; states=[]; acc=0.; prev=None
    for t in range(len(z)):
        evid=[]; signed=[]
        for b,ix in feature_blocks.items():
            avail=m[t,ix]>0
            if not avail.any() or not np.isfinite(null_q[b]): continue
            dev=float(np.sqrt(np.mean(z[t,ix][avail]**2)))
            p=(1+np.sum(np.asarray(null_scores[b])>=dev))/(1+len(null_scores[b]))
            evid.append(-np.log(max(p,1e-8))); signed.extend(z[t,ix][avail].tolist())
        if not evid:
            states.append("Unqualified"); score.append(np.nan); prev=None; acc=0.; continue
        coherence=abs(np.mean(np.sign(signed))) if signed else 0.
        dt=1 if prev is None else max(1,(t-prev))
        acc=acc*np.exp(-np.log(2)*dt/7)+max(0,float(np.sum(evid))*coherence-1.)
        state="Persistent" if acc>=8 else ("Emerging" if acc>=3 else "Stable")
        states.append(state); score.append(acc); prev=t
    return pd.DataFrame({"score":score,"state":states})
pqmd={u:pqmd_run(u) for u in Z}

## 16. DGCP: gated fast/slow personal residuals, trust region and verification

In [ ]:
# Prequential local personalization: forecast first, then update a bounded feature-wise fast adapter.
def dgcp_personalize(uid, model, data, pq):
    x,y,m=data; fast=np.zeros(NF,dtype="float32"); slow=np.zeros(NF,dtype="float32")
    pred=[]; truth=[]; masks=[]; gates=[]; states=pq["state"].tolist()
    for i in range(len(y)):
        base=model([x[0][i:i+1],x[1][i:i+1]],training=False).numpy()[0]
        state=states[min(i+1,len(states)-1)]
        gate=1. if state=="Persistent" else (.5 if state=="Emerging" else 0.)
        forecast=base+slow+fast
        pred.append(forecast.copy()); truth.append(y[i]); masks.append(m[i]); gates.append(gate)
        err=(y[i]-forecast)*m[i]
        # Bounded fast residual update, only after the current target is observed.
        if gate>0:
            fast += .05*gate*err
            norm=np.linalg.norm(fast)
            if norm>2.: fast*=2./norm
        else: fast*=.9
        # Candidate consolidation requires persistent state and stabilization proxy.
        if state=="Persistent" and i>=7:
            recent=np.asarray(pred[-7:])
            stable=np.mean(np.std(recent,axis=0))<.75
            if stable:
                candidate=slow+fast
                # Chronological frozen comparison: compare last eight errors if available.
                if len(pred)>=8:
                    cand=np.mean(np.abs(np.asarray(truth[-8:])-(np.asarray(pred[-8:])+fast))*np.asarray(masks[-8:]))
                    base_loss=np.mean(np.abs(np.asarray(truth[-8:])-np.asarray(pred[-8:]))*np.asarray(masks[-8:]))
                    if cand<base_loss: slow=candidate; fast[:]=0.
    return np.asarray(pred),np.asarray(truth),np.asarray(masks),np.asarray(gates)
ph_results={}
for uid,data in ph_data.items():
    ph_results[uid]=dgcp_personalize(uid,ph_model,data,pqmd[uid])

## 17. PerHeart prequential forecasting evaluation and uncertainty

In [ ]:
rows=[]; residuals={i:[] for i in range(NF)}
for uid,(pred,true,mask,gate) in ph_results.items():
    for j in range(NF):
        ok=mask[:,j]>0
        if ok.any():
            residuals[j].extend(np.abs(true[ok,j]-pred[ok,j]).tolist())
    err=np.abs(true-pred)*mask
    rows.append({"participant":uid,"observed_targets":int(mask.sum()),
                 "MAE_protected":float(err.sum()/max(mask.sum(),1)),
                 "RMSE_protected":float(np.sqrt(((true-pred)**2*mask).sum()/max(mask.sum(),1))),
                 "persistent_fraction":float(np.mean(gate>0))})
ph_metrics=pd.DataFrame(rows)
display(ph_metrics)
# Participant-balanced empirical 90% absolute-residual intervals; require >=5 participants per feature.
feature_intervals={}
for j in range(NF):
    vals=residuals[j]
    if len(vals)>=20 and len([u for u,r in ph_results.items() if (r[2][:,j]>0).any()])>=5:
        q=float(np.quantile(vals,.90,method="higher"))
        feature_intervals[PH_FEATURES[j]]=q
print("Calibrated feature intervals:",len(feature_intervals),"/",NF)

## 18. Wrist inertial data ingestion (when available)

In [ ]:
# The PerHeart archive stores wrist captures as Parquet. Extract compact daily features;
# no exact 50-Hz resampling is performed unless timestamps support it.
wrist_files=list(pathlib.Path(PERHEART_DIR).rglob("*.parquet"))
print("Wrist parquet files:",len(wrist_files))
wrist_daily=[]
for p in wrist_files:
    try: d=pd.read_parquet(p)
    except Exception: continue
    if d.empty: continue
    # User id is usually encoded in directory/file naming (user_1, user1, etc.).
    match=re.search(r"user[_-]?(\d+)",str(p),re.I)
    uid=match.group(1) if match else None
    if uid is None: continue
    cols={norm(c):c for c in d.columns}
    axes=[cols[k] for k in ["ax","ay","az","gx","gy","gz"] if k in cols]
    if len(axes)<6: continue
    a=d[[cols[k] for k in ["ax","ay","az"]]].to_numpy(float)
    g=d[[cols[k] for k in ["gx","gy","gz"]]].to_numpy(float)
    wrist_daily.append({"participant":uid,"file":p.name,"n_samples":len(d),
                        "acc_norm_mean":float(np.linalg.norm(a,axis=1).mean()),
                        "acc_norm_std":float(np.linalg.norm(a,axis=1).std()),
                        "gyro_norm_mean":float(np.linalg.norm(g,axis=1).mean()),
                        "gyro_norm_std":float(np.linalg.norm(g,axis=1).std())})
wrist_features=pd.DataFrame(wrist_daily)
display(wrist_features.head())

## 19. Statistical summaries, plots and saved artifacts

In [ ]:
os.makedirs(os.path.join(ROOT,"results"),exist_ok=True)
music_oof.to_csv(os.path.join(ROOT,"results","music_oof_predictions.csv"),index=False)
music_metrics.to_csv(os.path.join(ROOT,"results","music_metrics.csv"),index=False)
ph_metrics.to_csv(os.path.join(ROOT,"results","perheart_prequential_metrics.csv"),index=False)
ph_history.to_csv(os.path.join(ROOT,"results","perheart_federated_training.csv"),index=False)
wrist_features.to_csv(os.path.join(ROOT,"results","perheart_wrist_daily_features.csv"),index=False)
with pd.ExcelWriter(os.path.join(ROOT,"results","federated_heart_failure_results.xlsx")) as writer:
    music_metrics.to_excel(writer,sheet_name="MUSIC metrics",index=False)
    music_oof.to_excel(writer,sheet_name="MUSIC OOF",index=False)
    ph_metrics.to_excel(writer,sheet_name="PerHeart forecast",index=False)
    ph_history.to_excel(writer,sheet_name="PerHeart FL",index=False)
    wrist_features.to_excel(writer,sheet_name="Wrist features",index=False)
fig,ax=plt.subplots(1,2,figsize=(13,4))
sns.barplot(data=music_metrics,x="cause",y="AUROC",ax=ax[0]); ax[0].set_title("MUSIC cause-wise AUROC")
sns.barplot(data=ph_metrics,x="participant",y="MAE_protected",ax=ax[1]); ax[1].tick_params(axis="x",rotation=90); ax[1].set_title("PerHeart participant-wise MAE")
plt.tight_layout(); plt.savefig(os.path.join(ROOT,"results","cohort_performance.png"),dpi=300); plt.show()
print("Saved results to:",os.path.join(ROOT,"results"))

## 20. Interpretation and limitations

1. **MUSIC:** use the verified event-code dictionary before interpreting cause-specific risk. The notebook's four-year labels implement the manuscript's stated mapping only when the source cause field matches those codes. The provided public source's precise variable dictionary takes precedence.
2. **PerHeart:** no deterioration labels exist; results are next-observation forecasting and unsupervised drift evidence, not clinical deterioration prediction.
3. **Federation:** clients are simulated from participant/patient partitions. This is not evidence of deployment across real hospitals or devices.
4. **Privacy:** federated parameter exchange is not secure aggregation or differential privacy. No formal privacy guarantee is provided.
5. **PQMD/DGCP:** thresholds and consolidation gates are operational research settings and require inner-fold/participant-held-out calibration for confirmatory experiments. The notebook exposes these settings for sensitivity analysis.
6. **Uncertainty:** empirical residual ranges are not finite-sample conformal guarantees under temporal dependence.
7. **Clinical use:** this is a research implementation, not a medical device or a basis for diagnosis or treatment.

## Dataset citations

- MUSIC (Sudden Cardiac Death in Chronic Heart Failure), PhysioNet Open Data on AWS, DOI: [10.13026/z3m7-rf58](https://doi.org/10.13026/z3m7-rf58).
- Kolakowski et al. (2026). *Multimodal Dataset of In-Home Physiological and Inertial Measurements from Older Heart Failure Patients*. Data, 11(5), 106. DOI: [10.3390/data11050106](https://doi.org/10.3390/data11050106). Zenodo: [10.5281/zenodo.17143199](https://doi.org/10.5281/zenodo.17143199).